In [1]:
import sys
print(sys.executable)

c:\Users\NANDINI\Documents\ANACONDA\python.exe


In [2]:
from ultralytics import YOLO

print("✅ Ultralytics installed successfully!")

✅ Ultralytics installed successfully!


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA available: False
Using CPU


## Fish CSV → YOLO conversion

In [4]:
from pathlib import Path
import pandas as pd
import shutil

# =========================================================
# FISH DATASET
# =========================================================

SOURCE_DIR = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\Fish Diseases.v1i.tensorflow"
)

OUTPUT_DIR = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo"
)

# Exact class mapping from YOUR CSV
CLASS_MAP = {
    "Healthy Fish": 0,
    "Bacterial Diseases": 1,
    "Parasitic Diseases": 2,
    "White Tail Diseases": 3,
    "Fungal Diseases": 4
}

# Create YOLO folders
for split in ["train", "valid", "test"]:
    (OUTPUT_DIR / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )
    (OUTPUT_DIR / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )

# =========================================================
# CONVERT TRAIN / VALID / TEST
# =========================================================

for split in ["train", "valid", "test"]:

    csv_path = SOURCE_DIR / split / "_annotations.csv"

    print("\n================================")
    print("Processing:", split)
    print("================================")

    if not csv_path.exists():
        print("❌ CSV not found:", csv_path)
        continue

    df = pd.read_csv(csv_path)

    print("Annotations:", len(df))

    # Check required columns
    required_columns = [
        "filename",
        "width",
        "height",
        "class",
        "xmin",
        "ymin",
        "xmax",
        "ymax"
    ]

    missing = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {csv_path}: {missing}"
        )

    # Check classes
    unknown_classes = set(df["class"].astype(str)) - set(CLASS_MAP)

    if unknown_classes:
        raise ValueError(
            f"Unknown classes found: {unknown_classes}"
        )

    # Process every image
    for filename, group in df.groupby("filename"):

        source_image = SOURCE_DIR / split / filename

        if not source_image.exists():

            # Case-insensitive fallback
            matches = list(
                (SOURCE_DIR / split).glob(
                    Path(filename).stem + ".*"
                )
            )

            if matches:
                source_image = matches[0]
            else:
                print("⚠️ Image missing:", filename)
                continue

        # Copy image
        destination_image = (
            OUTPUT_DIR / split / "images" / source_image.name
        )

        shutil.copy2(
            source_image,
            destination_image
        )

        # Image size
        width = float(group.iloc[0]["width"])
        height = float(group.iloc[0]["height"])

        label_lines = []

        for _, row in group.iterrows():

            xmin = float(row["xmin"])
            ymin = float(row["ymin"])
            xmax = float(row["xmax"])
            ymax = float(row["ymax"])

            # Convert Pascal VOC bbox → YOLO bbox
            x_center = ((xmin + xmax) / 2) / width
            y_center = ((ymin + ymax) / 2) / height

            box_width = (xmax - xmin) / width
            box_height = (ymax - ymin) / height

            class_id = CLASS_MAP[
                str(row["class"])
            ]

            label_lines.append(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

        # Save YOLO label
        label_path = (
            OUTPUT_DIR
            / split
            / "labels"
            / f"{Path(filename).stem}.txt"
        )

        label_path.write_text(
            "\n".join(label_lines),
            encoding="utf-8"
        )

    image_count = len(
        list(
            (OUTPUT_DIR / split / "images").glob("*")
        )
    )

    label_count = len(
        list(
            (OUTPUT_DIR / split / "labels").glob("*.txt")
        )
    )

    print("✅ Images:", image_count)
    print("✅ Labels:", label_count)


# =========================================================
# CREATE data.yaml
# =========================================================

yaml_text = """path: C:/Users/NANDINI/Desktop/AquaCare_Complete/fish/fish_yolo

train: train/images
val: valid/images
test: test/images

nc: 5

names:
  0: Healthy Fish
  1: Bacterial Diseases
  2: Parasitic Diseases
  3: White Tail Diseases
  4: Fungal Diseases
"""

yaml_path = OUTPUT_DIR / "data.yaml"

yaml_path.write_text(
    yaml_text,
    encoding="utf-8"
)

print("\n================================")
print("✅ FISH YOLO DATASET READY")
print("================================")
print("Location:", OUTPUT_DIR)
print("YAML:", yaml_path)


Processing: train
Annotations: 1167
✅ Images: 722
✅ Labels: 722

Processing: valid
Annotations: 198
✅ Images: 96
✅ Labels: 96

Processing: test
Annotations: 233
✅ Images: 96
✅ Labels: 96

✅ FISH YOLO DATASET READY
Location: C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo
YAML: C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml


## Check Converted dataset

In [5]:
from pathlib import Path

for split in ["train", "valid", "test"]:

    images = list(
        (OUTPUT_DIR / split / "images").glob("*")
    )

    labels = list(
        (OUTPUT_DIR / split / "labels").glob("*.txt")
    )

    print(
        split,
        "→ images:", len(images),
        "| labels:", len(labels)
    )

print("\n✅ data.yaml:")
print(yaml_path.read_text())

train → images: 722 | labels: 722
valid → images: 96 | labels: 96
test → images: 96 | labels: 96

✅ data.yaml:
path: C:/Users/NANDINI/Desktop/AquaCare_Complete/fish/fish_yolo

train: train/images
val: valid/images
test: test/images

nc: 5

names:
  0: Healthy Fish
  1: Bacterial Diseases
  2: Parasitic Diseases
  3: White Tail Diseases
  4: Fungal Diseases



## Train Fish model

In [10]:
from ultralytics import YOLO

fish_model = YOLO("yolo11n.pt")

results = fish_model.train(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml",
    epochs=5,
    imgsz=256,
    batch=2,
    workers=0,
    device="cpu",
    fraction=0.15,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models",
    name="fish_fast_test",
    exist_ok=True
)

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.15, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fish_fas

In [6]:
from pathlib import Path
from ultralytics import YOLO

# =========================================================
# FISH BEST MODEL FIND
# =========================================================

fish_models_dir = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models"
)

possible_models = [
    fish_models_dir / "fish_disease" / "weights" / "best.pt",
    fish_models_dir / "fish_disease_final" / "weights" / "best.pt",
    fish_models_dir / "fish_disease_final_fast" / "weights" / "best.pt",
    fish_models_dir / "fish_fast_test" / "weights" / "best.pt",
]

best_model_path = None

for p in possible_models:
    if p.exists():
        best_model_path = p
        break

if best_model_path is None:
    all_best = list(fish_models_dir.rglob("best.pt"))

    if all_best:
        best_model_path = all_best[0]

if best_model_path is None:
    raise FileNotFoundError(
        "❌ Fish best.pt not found inside the models folder."
    )

print("✅ Fish model found:")
print(best_model_path)


# =========================================================
# LOAD MODEL
# =========================================================

model = YOLO(str(best_model_path))


# =========================================================
# TEST VALIDATION
# =========================================================

data_yaml = r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml"

metrics = model.val(
    data=data_yaml,
    split="test",
    plots=True
)


# =========================================================
# RESULTS
# =========================================================

print("\n======================================")
print("🐟 FISH YOLO TEST RESULTS")
print("======================================")

print("mAP50    :", round(metrics.box.map50, 4))
print("mAP50 %  :", round(metrics.box.map50 * 100, 2), "%")

print("mAP50-95 :", round(metrics.box.map, 4))
print("mAP50-95%:", round(metrics.box.map * 100, 2), "%")

✅ Fish model found:
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease\weights\best.pt
Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,583,127 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.50.1 ms, read: 77.652.1 MB/s, size: 59.6 KB)
val: Scanning C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\labels... 96 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 467.8it/s 0.2s0.0s
val: New cache created: C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2it/s 4.9s0.9ss
                   all         96        233      0.481      0.333      0.294      0.182
          Healthy Fish         60        160      0.523      0.812      0.736      0.502
    Bacterial Diseases         16         16      0.318      0.

In [7]:
names = model.names

print("\n======================================")
print("🐟 CLASS-WISE RESULTS")
print("======================================")

for i, class_name in enumerate(names):
    try:
        print(
            f"{i} - {class_name}: "
            f"Precision={metrics.box.p[i]:.3f}, "
            f"Recall={metrics.box.r[i]:.3f}, "
            f"mAP50={metrics.box.ap50[i]:.3f}"
        )
    except Exception:
        pass


🐟 CLASS-WISE RESULTS
0 - 0: Precision=0.523, Recall=0.812, mAP50=0.736
1 - 1: Precision=0.318, Recall=0.688, mAP50=0.568
2 - 2: Precision=0.400, Recall=0.120, mAP50=0.136
3 - 3: Precision=1.000, Recall=0.000, mAP50=0.003
4 - 4: Precision=0.165, Recall=0.043, mAP50=0.027


In [8]:
from pathlib import Path

last = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease_final\weights\last.pt"
)

print("Last model exists:", last.exists())
print("Path:", last)

Last model exists: False
Path: C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease_final\weights\last.pt


In [6]:
from ultralytics import YOLO

fish_model = YOLO("yolo11n.pt")

results = fish_model.train(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml",
    epochs=30,
    imgsz=224,
    batch=2,
    workers=0,
    device="cpu",
    fraction=1.0,
    patience=5,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models",
    name="fish_disease_final_fast",
    exist_ok=True,
    plots=False
)

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fish_dis

In [9]:
from pathlib import Path

best = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_fast_test\weights\best.pt"
)

print("Best model exists:", best.exists())
print("Path:", best)

Best model exists: True
Path: C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_fast_test\weights\best.pt


In [10]:
from ultralytics import YOLO

fish_model = YOLO(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease\weights\best.pt"
)

metrics = fish_model.val(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml",
    split="test",
    plots=True
)

print("====================================")
print("🐟 FISH MODEL TEST RESULTS")
print("====================================")
print("mAP50    :", round(metrics.box.map50, 4))
print("mAP50-95 :", round(metrics.box.map, 4))

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,583,127 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.40.0 ms, read: 116.239.8 MB/s, size: 55.7 KB)
val: Scanning C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\labels.cache... 96 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 18.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.7s1.0ss
                   all         96        233      0.481      0.333      0.294      0.182
          Healthy Fish         60        160      0.523      0.812      0.736      0.502
    Bacterial Diseases         16         16      0.318      0.688      0.568      0.346
    Parasitic Diseases          5         25        0.4       0.12      0.136     0.0555
   White Tail Diseases          9          9          1          0    0.00335    0.001

## fish model

In [11]:
from pathlib import Path

fish_models = list(
    Path(r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models").rglob("best.pt")
)

print("🐟 Fish models:")
for p in fish_models:
    print(p)

🐟 Fish models:
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_fast_test\weights\best.pt
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease_final_fast\weights\best.pt
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease-2\weights\best.pt
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease\weights\best.pt


In [12]:
results = fish_model.predict(
    source=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images",
    conf=0.25,
    save=True,
    project=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models",
    name="fish_test_predictions",
    exist_ok=True
)

print("✅ Fish test prediction completed")


image 1/96 C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images\bacterial_diseases_koi_002_jpg.rf.8dd1af9266169f4a8392cc328ca2e1ef.jpg: 320x320 1 Bacterial Diseases, 179.2ms
image 2/96 C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images\bacterial_diseases_koi_010_jpg.rf.c7ac777cd0b74dce4669d7b4acf56cbf.jpg: 320x320 4 Bacterial Diseasess, 98.5ms
image 3/96 C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images\bacterial_diseases_koi_011_jpg.rf.37739914d39de08b86ec82e477efd922.jpg: 320x320 (no detections), 70.3ms
image 4/96 C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images\bacterial_diseases_koi_021_jpg.rf.8f481fea019570b6bbde90c769041cbf.jpg: 320x320 1 Healthy Fish, 4 Bacterial Diseasess, 129.4ms
image 5/96 C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\images\bacterial_diseases_koi_024_jpg.rf.af7059660d13ff89c0b5f4fcb8022e5d.jpg: 320x320 1 Healthy Fish, 3 Bacterial Diseasess, 106.3ms
image 6/96 C:\Us

In [13]:
from pathlib import Path

result_folder = Path(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions"
)

print("Prediction folder:")
print(result_folder)

print("\nImages:")
for img in list(result_folder.glob("*"))[:10]:
    print(img)

Prediction folder:
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions

Images:
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_002_jpg.rf.8dd1af9266169f4a8392cc328ca2e1ef.jpg
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_010_jpg.rf.c7ac777cd0b74dce4669d7b4acf56cbf.jpg
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_011_jpg.rf.37739914d39de08b86ec82e477efd922.jpg
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_021_jpg.rf.8f481fea019570b6bbde90c769041cbf.jpg
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_024_jpg.rf.af7059660d13ff89c0b5f4fcb8022e5d.jpg
C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_test_predictions\bacterial_diseases_koi_026_jpg.rf.183e512ce974d5d7b6aa3d318367f2d1.jpg
C:\User

## Test the trained model

In [14]:
from ultralytics import YOLO

model = YOLO(
    r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\models\fish_disease\weights\best.pt"
)

metrics = model.val(
    data=r"C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\data.yaml",
    split="test"
)

print(metrics)

Ultralytics 8.4.171  Python-3.13.9 torch-2.14.1+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,583,127 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 91.638.9 MB/s, size: 44.2 KB)
val: Scanning C:\Users\NANDINI\Desktop\AquaCare_Complete\fish\fish_yolo\test\labels.cache... 96 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 15.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2s/it 7.4s1.4ss
                   all         96        233      0.481      0.333      0.294      0.182
          Healthy Fish         60        160      0.523      0.812      0.736      0.502
    Bacterial Diseases         16         16      0.318      0.688      0.568      0.346
    Parasitic Diseases          5         25        0.4       0.12      0.136     0.0555
   White Tail Diseases          9          9          1          0    0.00335    0.0010